# Trail Popularity Prediction - Final Analysis
## Streamlined: Top 150 Features by Importance Only
Single notebook using the optimal feature selection method

In [1]:
# CELL 1: IMPORTS & SETUP
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
import os
from math import radians, sin, cos, sqrt, atan2

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ All imports successful")

✓ All imports successful


/Users/drapcat/4380env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# CELL 2: HAVERSINE FUNCTION FOR METRO FEATURES
def haversine(lat1, lon1, lat2, lon2):
    """Calculate haversine distance between two points in km."""
    R = 6371  # Earth's radius in km
    
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    
    return R * c

print("✓ Haversine function loaded")

✓ Haversine function loaded


In [3]:
# CELL 3: COMPLETE PREPROCESSING PIPELINE
print("="*80)
print("STEP 1: LOAD & PREPROCESS DATA")
print("="*80)

# Load data
df = pd.read_csv('./Data/preprocessed_trails.csv')
print(f"\nDataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

# STEP 1: Prepare features
y = df['popularity'].copy()
X = df.drop(columns=['popularity']).copy()

print(f"Target (y): {y.shape[0]:,} samples")
print(f"Features (X): {X.shape}")
print(f"\nTarget statistics:")
print(f"  Mean: {y.mean():.2f}, Median: {y.median():.2f}, Std: {y.std():.2f}")

# STEP 2: Encode categorical features
print("\n" + "-"*80)
print("ENCODING CATEGORICAL FEATURES")
print("-"*80)

# routeType: Ordinal encoding
if 'routeType' in X.columns:
    route_map = {'P': 1, 'O': 2, 'L': 3}
    X['routeType_encoded'] = X['routeType'].map(route_map)
    X = X.drop(columns=['routeType'])
    print("✓ Ordinal encoded routeType (P=1, O=2, L=3)")

# Convert boolean columns to int
bool_cols = X.select_dtypes(include=['bool']).columns
for col in bool_cols:
    X[col] = X[col].astype(int)

if len(bool_cols) > 0:
    print(f"✓ Converted {len(bool_cols)} boolean columns to int (0/1)")

# One-hot encode categorical columns
categorical_cols = ['cityName', 'stateName', 'areaType']

for col in categorical_cols:
    if col in X.columns:
        X[col] = X[col].fillna('Unknown')
        encoded = pd.get_dummies(X[col], prefix=col, drop_first=True)
        X = pd.concat([X, encoded], axis=1)
        X = X.drop(columns=[col])
        print(f"✓ One-hot encoded {col} ({len(encoded.columns)} columns)")

print(f"\nFeature shape after encoding: {X.shape}")

# STEP 3: Create metro features
print("\n" + "-"*80)
print("CREATING METRO FEATURES")
print("-"*80)

metro_path = './Data/usmetros.csv'
if os.path.exists(metro_path) and 'latitude' in X.columns and 'longitude' in X.columns:
    try:
        metros = pd.read_csv(metro_path)
        metros.rename(columns={'lat': 'metro_lat', 'lng': 'metro_lng'}, inplace=True)
        
        distances = []
        nearest_pop = []
        pop_sum_3 = []
        
        metro_coords = metros[['metro_lat', 'metro_lng']].values
        metro_pops = metros['population'].values
        
        print(f"Calculating distances to {len(metros)} metros...")
        
        for idx, row in X.iterrows():
            if idx % 500 == 0:
                print(f"  Progress: {idx} / {len(X)}")
            
            lat, lon = row['latitude'], row['longitude']
            
            dists = np.array([
                haversine(lat, lon, mlat, mlon)
                for (mlat, mlon) in metro_coords
            ])
            
            sorted_idx = np.argsort(dists)
            distances.append(dists[sorted_idx[0]])
            nearest_pop.append(metro_pops[sorted_idx[0]])
            pop_sum_3.append(metro_pops[sorted_idx[:3]].sum())
        
        X['dist_to_nearest_metro_km'] = distances
        X['nearest_metro_population'] = nearest_pop
        X['sum_population_3_nearest_metros'] = pop_sum_3
        
        print("✓ Created 3 metro-based features")
    except Exception as e:
        print(f"⚠ Could not create metro features: {e}")
else:
    print("⚠ Metro file not found or lat/lon missing - skipping metro features")

# STEP 4: Remove location and categorical columns
print("\n" + "-"*80)
print("REMOVING LOCATION & CATEGORICAL COLUMNS")
print("-"*80)

cols_to_remove = ['trail_id', 'latitude', 'longitude', 'trailType']
cols_to_drop = [col for col in cols_to_remove if col in X.columns]

if cols_to_drop:
    X = X.drop(columns=cols_to_drop)
    print(f"✓ Dropped {len(cols_to_drop)} location columns: {cols_to_drop}")

# STEP 5: Handle missing values
print("\n" + "-"*80)
print("HANDLING MISSING VALUES")
print("-"*80)

missing_counts = X.isnull().sum()
cols_with_missing = missing_counts[missing_counts > 0]

if len(cols_with_missing) > 0:
    print(f"Found {len(cols_with_missing)} columns with missing values:")
    for col, count in cols_with_missing.items():
        pct = (count / len(X)) * 100
        print(f"  - {col}: {count} ({pct:.2f}%)")
        if X[col].dtype in ['int64', 'float64']:
            median = X[col].median()
            X[col] = X[col].fillna(median)
else:
    print("✓ No missing values found")

print(f"\nFinal preprocessed dataset: {X.shape}")
print(f"Feature dtypes check:")
print(X.dtypes.value_counts())

STEP 1: LOAD & PREPROCESS DATA

Dataset loaded: 4,565 rows, 93 columns
Target (y): 4,565 samples
Features (X): (4565, 92)

Target statistics:
  Mean: 77.31, Median: 83.95, Std: 21.71

--------------------------------------------------------------------------------
ENCODING CATEGORICAL FEATURES
--------------------------------------------------------------------------------
✓ Ordinal encoded routeType (P=1, O=2, L=3)
✓ Converted 7 boolean columns to int (0/1)
✓ One-hot encoded cityName (415 columns)
✓ One-hot encoded stateName (7 columns)
✓ One-hot encoded areaType (14 columns)

Feature shape after encoding: (4565, 525)

--------------------------------------------------------------------------------
CREATING METRO FEATURES
--------------------------------------------------------------------------------
Calculating distances to 387 metros...
  Progress: 0 / 4565
  Progress: 500 / 4565
  Progress: 1000 / 4565
  Progress: 1500 / 4565
  Progress: 2000 / 4565
  Progress: 2500 / 4565
  Progr

In [4]:
# TRAIN/TEST SPLIT
print("\n" + "="*80)
print("STEP 2: TRAIN/TEST SPLIT")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\n✓ Training set: {len(X_train):,} samples, {X_train.shape[1]} features")
print(f"✓ Testing set: {len(X_test):,} samples, {X_test.shape[1]} features")
print(f"\nAll features are numeric: {X_train.select_dtypes(include=[np.number]).shape[1] == X_train.shape[1]}")


STEP 2: TRAIN/TEST SPLIT

✓ Training set: 3,652 samples, 525 features
✓ Testing set: 913 samples, 525 features

All features are numeric: False


In [5]:
# FEATURE SELECTION - TOP 150 BY IMPORTANCE
print("\n" + "="*80)
print("STEP 3: FEATURE SELECTION - TOP 150 BY IMPORTANCE")
print("="*80)

# Train baseline to get feature importances
print("\nTraining baseline model to rank features...")
baseline = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    verbosity=0,
    n_jobs=-1
)
baseline.fit(X_train, y_train)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': baseline.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 20 most important features:")
for idx, row in feature_importance.head(20).iterrows():
    print(f"  {row['feature']:<40} {row['importance']:.6f}")

# Select top 150 features
top_150_features = feature_importance.head(150)['feature'].tolist()
print(f"\n✓ Selected top 150 features for final model")
print(f"✓ These {len(top_150_features)} features account for 71.4% reduction")


STEP 3: FEATURE SELECTION - TOP 150 BY IMPORTANCE

Training baseline model to rank features...

Top 20 most important features:
  isClosed                                 0.183717
  numFeaturedPhotos_pct                    0.126017
  collections_trending                     0.063080
  avgRating                                0.058882
  collections_with_photos_pct              0.030747
  numPOIs                                  0.025845
  numRecordings_pct                        0.022060
  activities_walking                       0.017739
  kidFriendly                              0.013874
  activities_backpacking                   0.013074
  elevationGainMeters                      0.011506
  estimatedTime                            0.010485
  areaType_W                               0.009851
  lengthMeters                             0.009572
  cityName_Indian Hills                    0.009397
  dist_to_nearest_metro_km                 0.008910
  stateName_Colorado                   

In [11]:
# TRAIN FINAL MODEL ON TOP 150 FEATURES
print("\n" + "="*80)
print("STEP 4: TRAINING FINAL XGBOOST MODEL (TOP 150 FEATURES)")
print("="*80)

# Select top 150 features from train/test
X_train_150 = X_train[top_150_features]
X_test_150 = X_test[top_150_features]

# Train final model
print(f"\nTraining final model on {len(top_150_features)} features...")
final_model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    verbosity=0,
    n_jobs=-1
)
final_model.fit(X_train_150, y_train)

# Evaluate
train_pred = final_model.predict(X_train_150)
test_pred = final_model.predict(X_test_150)

train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
test_mae = mean_absolute_error(y_test, test_pred)

print(f"\nFinal Model Performance:")
print(f"  Training R²: {train_r2:.4f}")
print(f"  Testing R²:  {test_r2:.4f}")
print(f"  Testing RMSE: {test_rmse:.4f}")
print(f"  Testing MAE:  {test_mae:.4f}")

print(f"\n✓ Model trained")

import pickle

# Save the trained model
with open('./Data/final_model_trained.pkl', 'wb') as f:
    pickle.dump(final_model, f)

print("✓ Model saved to ./Data/final_model_trained.pkl")


STEP 4: TRAINING FINAL XGBOOST MODEL (TOP 150 FEATURES)

Training final model on 150 features...

Final Model Performance:
  Training R²: 0.9710
  Testing R²:  0.8576
  Testing RMSE: 8.2038
  Testing MAE:  5.7233

✓ Model trained
✓ Model saved to ./Data/final_model_trained.pkl


In [7]:
# CELL 7: COMPUTE SHAP VALUES
print("\n" + "="*80)
print("STEP 5: SHAP EXPLAINABLE AI ANALYSIS")
print("="*80)

print("\nComputing SHAP values (this may take 5-10 minutes)...")

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test_150)

print(f"✓ SHAP values computed for {len(X_test_150)} test samples")


STEP 5: SHAP EXPLAINABLE AI ANALYSIS

Computing SHAP values (this may take 5-10 minutes)...
✓ SHAP values computed for 913 test samples


In [8]:
# SHAP SUMMARY & ANALYSIS
print("\n" + "="*80)
print("SHAP FEATURE IMPORTANCE RANKING")
print("="*80)

# Calculate mean absolute SHAP values
shap_importance = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    'Feature': top_150_features,
    'Mean_SHAP': shap_importance
}).sort_values('Mean_SHAP', ascending=False).reset_index(drop=True)

shap_df['Rank'] = range(1, len(shap_df) + 1)

print("\nTop 30 Features by SHAP:")
print(shap_df.head(30).to_string())


SHAP FEATURE IMPORTANCE RANKING

Top 30 Features by SHAP:
                            Feature  Mean_SHAP  Rank
0             numFeaturedPhotos_pct   5.378812     1
1                         avgRating   3.681537     2
2                          isClosed   3.497097     3
3       collections_with_photos_pct   2.549634     4
4              collections_trending   2.228457     5
5                           numPOIs   2.152720     6
6                 numRecordings_pct   1.557913     7
7                sentence_count_all   1.146005     8
8          dist_to_nearest_metro_km   1.050416     9
9                   sentiment_views   0.722673    10
10                    numPhotos_pct   0.688089    11
11                     lengthMeters   0.660120    12
12              elevationGainMeters   0.595881    13
13  sum_population_3_nearest_metros   0.543588    14
14                    sentiment_all   0.481379    15
15                        hasAlerts   0.443672    16
16                    estimatedTime   0.

In [9]:
# EXTRACT FORCE PLOT SAMPLES FOR APP TESTING
print("\n" + "="*80)
print("EXTRACTING FORCE PLOT SAMPLE VALUES FOR APP TESTING")
print("="*80)

# Make predictions
pred = final_model.predict(X_test_150)

# Get diverse sample indices
sample_indices = [
    np.argmax(pred),  # Highest prediction
    np.argmin(pred),  # Lowest prediction
    np.argsort(np.abs(pred - np.median(pred)))[0],  # Near median 1
    np.argsort(np.abs(pred - np.median(pred)))[1],  # Near median 2
    np.argsort(np.abs(pred - np.median(pred)))[2]   # Near median 3
]

print("\nForce Plot Samples (for testing in app):")
print("-"*80)

for sample_num, actual_idx in enumerate(sample_indices, 1):
    actual_pop = y_test.iloc[actual_idx]
    pred_pop = pred[actual_idx]
    sample_data = X_test_150.iloc[actual_idx]
    
    print(f"\nFORCE PLOT {sample_num}:")
    print(f"  Actual Popularity: {actual_pop:.2f}")
    print(f"  Predicted Popularity: {pred_pop:.2f}")
    print(f"  Features: {len(sample_data)}")
    
    # Save to CSV for easy use in app
    feature_dict = dict(sample_data)
    feature_dict['actual_popularity'] = actual_pop
    feature_dict['predicted_popularity'] = pred_pop
    
    df_sample = pd.DataFrame([feature_dict])
    csv_filename = f'force_plot_{sample_num}_features.csv'
    df_sample.to_csv(csv_filename, index=False)
    print(f"  ✓ Saved to: {csv_filename}")

print(f"\n✓ Force plot samples extracted")


EXTRACTING FORCE PLOT SAMPLE VALUES FOR APP TESTING

Force Plot Samples (for testing in app):
--------------------------------------------------------------------------------

FORCE PLOT 1:
  Actual Popularity: 99.82
  Predicted Popularity: 103.12
  Features: 150
  ✓ Saved to: force_plot_1_features.csv

FORCE PLOT 2:
  Actual Popularity: 15.80
  Predicted Popularity: 9.51
  Features: 150
  ✓ Saved to: force_plot_2_features.csv

FORCE PLOT 3:
  Actual Popularity: 90.04
  Predicted Popularity: 81.54
  Features: 150
  ✓ Saved to: force_plot_3_features.csv

FORCE PLOT 4:
  Actual Popularity: 74.23
  Predicted Popularity: 81.51
  Features: 150
  ✓ Saved to: force_plot_4_features.csv

FORCE PLOT 5:
  Actual Popularity: 81.81
  Predicted Popularity: 81.50
  Features: 150
  ✓ Saved to: force_plot_5_features.csv

✓ Force plot samples extracted


In [10]:
# SAVE TOP 150 FEATURE LIST FOR APP VERSION
print("\n" + "="*80)
print("SAVING TOP 150 FEATURES FOR APP")
print("="*80)

# Save feature list
feature_list_df = pd.DataFrame({
    'rank': range(1, 151),
    'feature_name': top_150_features
})

feature_list_df.to_csv('top_150_features.csv', index=False)
print(f"✓ Saved feature list to: top_150_features.csv")

# Also save as Python list for easy copy-paste
with open('top_150_features.txt', 'w') as f:
    f.write("top_150_features = [\n")
    for i, feat in enumerate(top_150_features):
        f.write(f"    '{feat}',\n")
    f.write("]")

print(f"✓ Saved feature list to: top_150_features.txt")

print(f"\nFinal feature list ({len(top_150_features)} features):")
for i, feat in enumerate(top_150_features[:20], 1):
    print(f"  {i:3d}. {feat}")
print(f"  ...")
print(f"  {len(top_150_features)}. {top_150_features[-1]}")


SAVING TOP 150 FEATURES FOR APP
✓ Saved feature list to: top_150_features.csv
✓ Saved feature list to: top_150_features.txt

Final feature list (150 features):
    1. isClosed
    2. numFeaturedPhotos_pct
    3. collections_trending
    4. avgRating
    5. collections_with_photos_pct
    6. numPOIs
    7. numRecordings_pct
    8. activities_walking
    9. kidFriendly
   10. activities_backpacking
   11. elevationGainMeters
   12. estimatedTime
   13. areaType_W
   14. lengthMeters
   15. cityName_Indian Hills
   16. dist_to_nearest_metro_km
   17. stateName_Colorado
   18. collections_hidden-gems
   19. hasAlerts
   20. sentiment_views
  ...
  150. cityName_Taylorsville
